In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    precision_recall_fscore_support,
    average_precision_score, brier_score_loss
)
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

# ========== 引入指定的 9 种模型 ==========
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

# ----------------------
# 1. 路径配置
# ----------------------
base_data_path = r'D:\学习工作\'
result_save_path = os.path.join(base_data_path, '结果输出')

if not os.path.exists(result_save_path):
    os.makedirs(result_save_path)
    print(f"已创建结果保存文件夹：{result_save_path}")

# ----------------------
# 2. 数据读取与预处理
# ----------------------
def read_csv_safe(path):
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='gbk')

print("正在读取数据...")
datasets = {
    'Train': read_csv_safe(os.path.join(base_data_path, '训练集.csv')),
    'Validation': read_csv_safe(os.path.join(base_data_path, '验证集.csv'))
}

# 训练集缺失值中位数，用于后续统一填充，防止验证集发生数据泄露
train_dropped = datasets['Train'].dropna(subset=['status'])
feature_medians = train_dropped.drop(columns=['status']).median()

processed_data = {}
for name, df in datasets.items():
    df_dropped = df.dropna(subset=['status'])
    df_clean = df_dropped.replace([np.inf, -np.inf], np.nan).fillna(feature_medians)
    processed_data[name] = df_clean

# 提取两个数据集的公共特征
all_features = [set(df.columns) for df in processed_data.values()]
common_features = sorted(list(set.intersection(*all_features) - {'status'}))

X_dict, y_dict = {}, {}
ss = StandardScaler()

# 提取特征与标签
X_train_raw = processed_data['Train'][common_features]
y_train_raw = processed_data['Train']['status'].astype(int)

X_val_raw = processed_data['Validation'][common_features]
y_val = processed_data['Validation']['status'].astype(int)

# 标准化：Fit 于原始训练集，Transform 两个数据集
X_train_scaled = ss.fit_transform(X_train_raw)
X_val_scaled = ss.transform(X_val_raw)

# ----------------------
# 3. SMOTE 平衡处理 (仅应用于训练集)
# ----------------------
print(f"原始训练集分布 -> 0: {np.sum(y_train_raw == 0)}, 1: {np.sum(y_train_raw == 1)}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train_raw)

print(f"SMOTE 后训练集分布 -> 0: {np.sum(y_train_resampled == 0)}, 1: {np.sum(y_train_resampled == 1)}\n")

# 保存处理后的特征与标签
X_dict['Train'] = X_train_resampled
y_dict['Train'] = y_train_resampled
X_dict['Validation'] = X_val_scaled
y_dict['Validation'] = y_val

# ----------------------
# 4. 模型定义
# ----------------------
best_params = {
    'XGBoost': {
        'subsample': 0.6, 'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.5,
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1
    },
    'KNN': {
        'weights': 'uniform', 'p': 1, 'n_neighbors': 15, 'n_jobs': -1
    },
    'GBDT': {
        'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 3, 'learning_rate': 0.01, 'random_state': 42
    },
    'ANN': {
        'learning_rate_init': 0.05, 'hidden_layer_sizes': (32,), 'alpha': 0.0001, 
        'max_iter': 500, 'early_stopping': True, 'random_state': 42
    },
    'RF': {
        'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 4, 
        'random_state': 42, 'n_jobs': -1
    },
    'LR': {
        'solver': 'liblinear', 'penalty': 'l2', 'C': 1, 'random_state': 42
    },
    'DT': {
        'min_samples_split': 20, 'min_samples_leaf': 1, 'max_depth': 4, 'random_state': 42
    },
    'SVM': {
        'kernel': 'linear',  'C': 1, 
        'probability': True, 'random_state': 42  
    },
    'LightGBM': {
        'num_leaves': 16, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05, 
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
}

models = {
    'XGBoost': xgb.XGBClassifier(**best_params['XGBoost']),
    'GBDT': GradientBoostingClassifier(**best_params['GBDT']),
    'LightGBM': lgb.LGBMClassifier(**best_params['LightGBM']),
    'RF': RandomForestClassifier(**best_params['RF']),
    'LR': LogisticRegression(**best_params['LR']),
    'DT': DecisionTreeClassifier(**best_params['DT']),
    'KNN': KNeighborsClassifier(**best_params['KNN']),
    'ANN': MLPClassifier(**best_params['ANN']),
    'SVM': SVC(**best_params['SVM'])
}

# ----------------------
# 5. 训练模型
# ----------------------
print("🚀 开始训练所有模型...")
trained_models = {}
for model_name, model in models.items():
    print(f"   -> 正在训练: {model_name}")
    model.fit(X_dict['Train'], y_dict['Train'])
    trained_models[model_name] = model
print("✅ 所有模型训练完成！\n")

# ----------------------
# 6. 单次评估核心函数 (含校准指标)
# ----------------------
def get_metrics_dict(y_true, y_probs, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
        if len(np.unique(y_true)) == 1:
            if y_true[0] == 0: tn = cm[0, 0]
            else: tp = cm[0, 0]

    sens = tp / (tp + fn) if (tp + fn) != 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    acc = accuracy_score(y_true, y_pred)
    youden = sens + spec - 1
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    
    auc = roc_auc_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else np.nan
    auprc = average_precision_score(y_true, y_probs) if len(np.unique(y_true)) > 1 else np.nan
    brier = brier_score_loss(y_true, y_probs)
    
    # 拟合校准截距和斜率
    y_probs_clipped = np.clip(y_probs, 1e-7, 1 - 1e-7)
    logits = np.log(y_probs_clipped / (1 - y_probs_clipped)).reshape(-1, 1)
    
    try:
        # 使用 C=1e10 模拟无惩罚逻辑回归，保证所有 sklearn 版本的兼容性
        lr_cal = LogisticRegression(C=1e10, solver='lbfgs', max_iter=200)
        lr_cal.fit(logits, y_true)
        cal_intercept = lr_cal.intercept_[0]
        cal_slope = lr_cal.coef_[0][0]
    except Exception:
        cal_intercept, cal_slope = np.nan, np.nan

    return {
        'AUC': auc,
        'AUPRC': auprc,
        'Brier_Score': brier,
        'Cal_Intercept (校准截距)': cal_intercept,
        'Cal_Slope (校准斜率)': cal_slope,
        'Sensitivity (灵敏度)': sens,
        'Specificity (特异度)': spec,
        'Youden_Index (约登指数)': youden,
        'F1_Score': f1,
        'Accuracy (准确度)': acc
    }

# ----------------------
# 7. 包含 1000 次 Bootstrap 的置信区间计算函数
# ----------------------
def calculate_metrics_with_bootstrap(X, y, model, model_name, dataset_name, n_bootstraps=1000):
    y_pred = model.predict(X)
    y_probs = model.predict_proba(X)[:, 1].clip(1e-7, 1 - 1e-7)
    y_true = np.array(y)
    
    # 1. 计算整体点估计值
    point_estimates = get_metrics_dict(y_true, y_probs, y_pred)
    
    # 2. 初始化 Bootstrap 容器
    bootstrapped_metrics = {key: [] for key in point_estimates.keys()}
    n_samples = len(y_true)
    
    # 3. Bootstrap 抽样计算
    for _ in range(n_bootstraps):
        indices = resample(np.arange(n_samples), replace=True)
        y_true_b = y_true[indices]
        y_probs_b = y_probs[indices]
        y_pred_b = y_pred[indices]
        
        # 跳过单一类别抽样，避免报错
        if len(np.unique(y_true_b)) < 2:
            continue
            
        metrics_b = get_metrics_dict(y_true_b, y_probs_b, y_pred_b)
        for k, v in metrics_b.items():
            if not np.isnan(v):
                bootstrapped_metrics[k].append(v)
                
    # 4. 汇总点估计与 95% CI (格式: Estimate (Lower-Upper))
    final_results = {'Model': model_name, 'Dataset': dataset_name}
    for k, v in point_estimates.items():
        if np.isnan(v):
            final_results[k] = "NaN"
        else:
            if len(bootstrapped_metrics[k]) > 0:
                lower = np.percentile(bootstrapped_metrics[k], 2.5)
                upper = np.percentile(bootstrapped_metrics[k], 97.5)
                final_results[k] = f"{v:.4f} ({lower:.4f}-{upper:.4f})"
            else:
                final_results[k] = f"{v:.4f} (NaN-NaN)"
                
    return final_results

# ----------------------
# 8. 遍历并保存最终指标
# ----------------------
all_results = []
print("📊 开始进行指标计算与 Bootstrap 置信区间估计 (约需数分钟)...")
for model_name, model in trained_models.items():
    print(f"   -> 正在评估: {model_name}")
    for dataset_name in ['Train', 'Validation']:  
        metrics = calculate_metrics_with_bootstrap(
            X=X_dict[dataset_name],
            y=y_dict[dataset_name],
            model=model,
            model_name=model_name,
            dataset_name=dataset_name,
            n_bootstraps=1000
        )
        all_results.append(metrics)

# 按标准列序输出
result_cols = [
    'Model', 'Dataset', 'AUC', 'AUPRC', 'Brier_Score',
    'Cal_Intercept (校准截距)', 'Cal_Slope (校准斜率)',
    'Sensitivity (灵敏度)', 'Specificity (特异度)', 
    'Youden_Index (约登指数)', 'F1_Score', 'Accuracy (准确度)'
]
result_df = pd.DataFrame(all_results)[result_cols]

csv_save_path = os.path.join(result_save_path, '9模型_2数据集_综合指标与95CI汇总.csv')
result_df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

print(f"\n🎉 任务完成！带有 95% 置信区间的全指标文件已保存至：\n{csv_save_path}")

# 控制台打印验证集排行榜
# 注：此时列的数据类型为字符串 '0.xxxx (0.xxxx-0.xxxx)'，可以直接进行按字符串首位的正常大小排序
pd.set_option('display.max_columns', None)
print("\n=== 🏆 【验证集】表现排行榜（按 AUC 降序排列） ===")
val_results = result_df[result_df['Dataset'] == 'Validation'].sort_values(by='AUC', ascending=False)
print(val_results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import matplotlib.font_manager as fm
import os

# ==========================================
# 9. 绘制 SCI 级别的高清 ROC 曲线 (仅保存 PDF)
# ==========================================
print("📈 开始绘制并保存 ROC 曲线 (PDF 格式)...")

# 统一设置全局字体为 Times New Roman (SCI 要求)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

# 配置 9 种模型使用的高对比度独立颜色（科研常用色系）
colors = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', 
    '#8491B4', '#91D1C2', '#DC0000', '#7E6148'
]

# 分别对训练集和验证集绘制
for dataset_name in ['Train', 'Validation']:
    # 建立 6x6 英寸的画布，分辨率设置为 300 dpi
    fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
    
    auc_data = [] # 用于记录数据，以便后续将图例按 AUC 大小排序
    
    # 计算并绘制各个模型
    for i, (model_name, model) in enumerate(trained_models.items()):
        X = X_dict[dataset_name]
        y = y_dict[dataset_name]
        
        # 提取预测概率
        y_probs = model.predict_proba(X)[:, 1]
        
        # 计算假阳性率、真阳性率和 AUC
        fpr, tpr, _ = roc_curve(y, y_probs)
        roc_auc = auc(fpr, tpr)
        
        # 绘制曲线
        line, = ax.plot(fpr, tpr, lw=1.5, color=colors[i])
        
        # 保存线段和标签用于图例排序
        label = f'{model_name} (AUC = {roc_auc:.4f})'
        auc_data.append((roc_auc, line, label))

    # 绘制对角线 (Random Guess)
    ax.plot([0, 1], [0, 1], color='black', lw=1.5, linestyle='--')

    # 按 AUC 降序排列图例
    auc_data.sort(key=lambda x: x[0], reverse=True)
    sorted_lines = [item[1] for item in auc_data]
    sorted_labels = [item[2] for item in auc_data]
    
    # 坐标轴与边界格式设置
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    
    # 设置图标题 (如果不想在图中显示标题而是放在论文图注中，可注释掉下面这行)
    ax.set_title(f'ROC Curves on {dataset_name} Set', fontsize=16, fontweight='bold')
    
    # 设置图例
    ax.legend(sorted_lines, sorted_labels, loc="lower right", fontsize=10, 
              frameon=True, edgecolor='black', fancybox=False)

    # 加粗图片外围边框 (符合传统 SCI 审美)
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    
    # 设置刻度参数
    ax.tick_params(axis='both', which='major', labelsize=12, width=1.5)

    plt.tight_layout()
    
    # 仅保存为矢量图 PDF
    pdf_path = os.path.join(result_save_path, f'ROC_Curve_{dataset_name}.pdf')
    plt.savefig(pdf_path, format='pdf', bbox_inches='tight')
    plt.close()

print(f"🎉 ROC 曲线绘制完成！PDF 格式矢量图已保存至: {result_save_path}")

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
import numpy as np

# ==========================================
# 10. 绘制 SCI 级别的 DCA 曲线与校准曲线 (仅保存 PDF)
# ==========================================
print("📈 开始绘制并保存 DCA 曲线与校准曲线 (PDF 格式)...")

# 统一设置全局字体为 Times New Roman
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False 

# 与 ROC 保持一致的高对比度独立颜色
colors = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', 
    '#8491B4', '#91D1C2', '#DC0000', '#7E6148'
]

# 定义 DCA 计算函数
def calculate_net_benefit(y_true, y_probs, thresholds):
    net_benefits = []
    N = len(y_true)
    for pt in thresholds:
        if pt == 1.0:
            net_benefits.append(0)
            continue
        preds = (y_probs >= pt).astype(int)
        tp = np.sum((preds == 1) & (y_true == 1))
        fp = np.sum((preds == 1) & (y_true == 0))
        nb = (tp / N) - (fp / N) * (pt / (1 - pt))
        net_benefits.append(nb)
    return np.array(net_benefits)

thresholds = np.linspace(0.01, 0.99, 100)

for dataset_name in ['Train', 'Validation']:
    X = X_dict[dataset_name]
    y = np.array(y_dict[dataset_name])
    
    # ---------------------------------------------------
    # A. 绘制 DCA (决策曲线)
    # ---------------------------------------------------
    fig_dca, ax_dca = plt.subplots(figsize=(6, 6), dpi=300)
    
    # 计算并绘制 Treat All 曲线
    prevalence = np.mean(y)
    treat_all_nb = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))
    ax_dca.plot(thresholds, treat_all_nb, color='gray', lw=1.5, linestyle=':', label='Treat All')
    
    # 绘制 Treat None 曲线
    ax_dca.plot(thresholds, np.zeros_like(thresholds), color='black', lw=1.5, linestyle='-', label='Treat None')
    
    for i, (model_name, model) in enumerate(trained_models.items()):
        y_probs = model.predict_proba(X)[:, 1]
        nb = calculate_net_benefit(y, y_probs, thresholds)
        ax_dca.plot(thresholds, nb, color=colors[i], lw=1.5, label=model_name)
    
    ax_dca.set_xlim([0.0, 1.0])
    # 动态设定 DCA Y 轴下限（略微低于 0）和上限（略高于患病率）
    ax_dca.set_ylim([-0.05, max(0.2, prevalence + 0.05)])
    
    ax_dca.set_xlabel('Threshold Probability', fontsize=14, fontweight='bold')
    ax_dca.set_ylabel('Net Benefit', fontsize=14, fontweight='bold')
    ax_dca.set_title(f'Decision Curve Analysis - {dataset_name}', fontsize=16, fontweight='bold')
    ax_dca.legend(loc="upper right", fontsize=10, frameon=True, edgecolor='black', fancybox=False)
    
    for spine in ax_dca.spines.values():
        spine.set_linewidth(1.5)
    ax_dca.tick_params(axis='both', which='major', labelsize=12, width=1.5)
    
    plt.tight_layout()
    pdf_path_dca = os.path.join(result_save_path, f'DCA_Curve_{dataset_name}.pdf')
    fig_dca.savefig(pdf_path_dca, format='pdf', bbox_inches='tight')
    plt.close(fig_dca)
    
    # ---------------------------------------------------
    # B. 绘制 Calibration (校准曲线)
    # ---------------------------------------------------
    fig_cal, ax_cal = plt.subplots(figsize=(6, 6), dpi=300)
    
    # 完美校准参考线
    ax_cal.plot([0, 1], [0, 1], color='black', lw=1.5, linestyle='--', label='Perfectly Calibrated')
    
    for i, (model_name, model) in enumerate(trained_models.items()):
        y_probs = model.predict_proba(X)[:, 1]
        # n_bins=10 是一般医学论文常用的分箱数
        prob_true, prob_pred = calibration_curve(y, y_probs, n_bins=10, strategy='quantile')
        ax_cal.plot(prob_pred, prob_true, marker='o', markersize=4, color=colors[i], lw=1.5, label=model_name)
    
    ax_cal.set_xlim([-0.02, 1.02])
    ax_cal.set_ylim([-0.02, 1.02])
    
    ax_cal.set_xlabel('Mean Predicted Probability', fontsize=14, fontweight='bold')
    ax_cal.set_ylabel('Fraction of Positives', fontsize=14, fontweight='bold')
    ax_cal.set_title(f'Calibration Curve - {dataset_name}', fontsize=16, fontweight='bold')
    ax_cal.legend(loc="lower right", fontsize=10, frameon=True, edgecolor='black', fancybox=False)
    
    for spine in ax_cal.spines.values():
        spine.set_linewidth(1.5)
    ax_cal.tick_params(axis='both', which='major', labelsize=12, width=1.5)
    
    plt.tight_layout()
    pdf_path_cal = os.path.join(result_save_path, f'Calibration_Curve_{dataset_name}.pdf')
    fig_cal.savefig(pdf_path_cal, format='pdf', bbox_inches='tight')
    plt.close(fig_cal)

print(f"🎉 DCA 曲线与校准曲线绘制完成！PDF 格式矢量图已保存至: {result_save_path}")

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import os

# ==========================================
# 11. 绘制 XGBoost 的 SHAP 特征重要性摘要图 (仅保存 PDF)
# ==========================================
print("📈 开始绘制并保存 XGBoost 模型的 SHAP 摘要图 (PDF 格式)...")

# 统一设置全局字体为 Times New Roman
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False 

# 提取训练好的 XGBoost 模型
xgb_model = trained_models['XGBoost']

# 初始化 SHAP TreeExplainer
# 注意：对于 XGBoost binary:logistic，shap_values 返回的是正类的 log-odds 影响
explainer = shap.TreeExplainer(xgb_model)

for dataset_name in ['Train', 'Validation']:
    # 将 NumPy 数组转回 DataFrame，并赋予特征列名，以便在图表中正确显示特征名称
    X_df = pd.DataFrame(X_dict[dataset_name], columns=common_features)
    
    # 计算 SHAP 值
    shap_values = explainer.shap_values(X_df)
    
    # ---------------------------------------------------
    # A. 绘制 SHAP 特征重要性条形图 (Bar Plot)
    # ---------------------------------------------------
    plt.figure()
    # plot_size=(7,6) 为特征名预留足够的左侧空间
    shap.summary_plot(shap_values, X_df, plot_type="bar", show=False, plot_size=(7, 6))
    
    ax_bar = plt.gca()
    ax_bar.set_xlabel('mean(|SHAP value|) (average impact on model output)', fontsize=12, fontweight='bold')
    ax_bar.set_title(f'SHAP Feature Importance (Bar) - {dataset_name}', fontsize=14, fontweight='bold')
    
    # 强制显示并加粗所有边框 (SHAP 默认会隐藏顶部和右侧边框)
    for spine in ax_bar.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
    ax_bar.tick_params(axis='both', which='major', labelsize=11, width=1.5)
    
    plt.tight_layout()
    pdf_path_bar = os.path.join(result_save_path, f'SHAP_Summary_Bar_XGBoost_{dataset_name}.pdf')
    plt.savefig(pdf_path_bar, format='pdf', bbox_inches='tight')
    plt.close()
    
    # ---------------------------------------------------
    # B. 绘制 SHAP 特征重要性蜂群图 (Beeswarm Plot)
    # ---------------------------------------------------
    plt.figure()
    shap.summary_plot(shap_values, X_df, show=False, plot_size=(7, 6))
    
    ax_bee = plt.gca()
    ax_bee.set_xlabel('SHAP value (impact on model output)', fontsize=12, fontweight='bold')
    ax_bee.set_title(f'SHAP Feature Importance (Beeswarm) - {dataset_name}', fontsize=14, fontweight='bold')
    
    # 强制显示并加粗所有边框
    for spine in ax_bee.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
    ax_bee.tick_params(axis='both', which='major', labelsize=11, width=1.5)
    
    plt.tight_layout()
    pdf_path_bee = os.path.join(result_save_path, f'SHAP_Summary_Beeswarm_XGBoost_{dataset_name}.pdf')
    plt.savefig(pdf_path_bee, format='pdf', bbox_inches='tight')
    plt.close()

print(f"🎉 XGBoost 的 SHAP 摘要图 (条形图 & 蜂群图) 绘制完成！PDF 已保存至: {result_save_path}")

In [ ]:
from sklearn.utils import resample

# ==========================================
# 13. 为敏感性分析结果计算 1000 次 Bootstrap 95% CI
# ==========================================
print("⏳ 正在进行 Bootstrap 重抽样计算 AUC 的 95% 置信区间 (约需几十秒)...")

def calculate_auc_with_ci(y_true, y_probs, n_bootstraps=1000):
    y_true = np.array(y_true)
    point_auc = roc_auc_score(y_true, y_probs)
    
    bootstrapped_aucs = []
    n_samples = len(y_true)
    
    for _ in range(n_bootstraps):
        indices = resample(np.arange(n_samples), replace=True)
        y_true_b = y_true[indices]
        y_probs_b = y_probs[indices]
        
        # 确保重抽样中至少包含正负两类样本
        if len(np.unique(y_true_b)) < 2:
            continue
            
        bootstrapped_aucs.append(roc_auc_score(y_true_b, y_probs_b))
        
    lower_bound = np.percentile(bootstrapped_aucs, 2.5)
    upper_bound = np.percentile(bootstrapped_aucs, 97.5)
    
    return point_auc, lower_bound, upper_bound

# 1. 计算【去除 D-dimer】模型的 AUC 及其 95% CI
train_auc_sens, train_lower_sens, train_upper_sens = calculate_auc_with_ci(y_train_raw, y_train_probs_sens)
val_auc_sens, val_lower_sens, val_upper_sens = calculate_auc_with_ci(y_val, y_val_probs_sens)

# 2. 计算【完整特征 (基线)】模型的 AUC 及其 95% CI
train_auc_base, train_lower_base, train_upper_base = calculate_auc_with_ci(y_train_raw, y_train_probs_base)
val_auc_base, val_lower_base, val_upper_base = calculate_auc_with_ci(y_val, y_val_probs_base)

# 3. 打印标准化对比结果
print("\n" + "="*65)
print(f"📊 敏感性分析最终结果 (XGBoost) - 包含 95% CI")
print("="*65)

print(f"【完整特征 (基线)】")
print(f"   -> 训练集 AUC: {train_auc_base:.4f} (95% CI: {train_lower_base:.4f} - {train_upper_base:.4f})")
print(f"   -> 验证集 AUC: {val_auc_base:.4f} (95% CI: {val_lower_base:.4f} - {val_upper_base:.4f})\n")

print(f"【去除 '{feature_to_remove}'】")
print(f"   -> 训练集 AUC: {train_auc_sens:.4f} (95% CI: {train_lower_sens:.4f} - {train_upper_sens:.4f})")
print(f"   -> 验证集 AUC: {val_auc_sens:.4f} (95% CI: {val_lower_sens:.4f} - {val_upper_sens:.4f})")
print("="*65)

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 12. 基于新数据集的 XGBoost 敏感性分析
# ==========================================
print("🔍 开始执行敏感性分析 (使用全新的敏感性数据集)...")

# 1. 路径配置与数据读取
base_data_path = r'D:\学习工作'

def read_csv_safe(path):
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='gbk')

print("📂 正在读取敏感性分析数据集...")
sens_datasets = {
    'Sens_Train': read_csv_safe(os.path.join(base_data_path, '敏感性训练集.csv')),
    'Sens_Validation': read_csv_safe(os.path.join(base_data_path, '敏感性验证集.csv'))
}

# 2. 数据清洗与特征预处理 (防止数据泄露)
# 使用新的敏感性训练集的中位数填充缺失值
train_dropped_sens = sens_datasets['Sens_Train'].dropna(subset=['status'])
feature_medians_sens = train_dropped_sens.drop(columns=['status']).median()

processed_sens_data = {}
for name, df in sens_datasets.items():
    df_dropped = df.dropna(subset=['status'])
    df_clean = df_dropped.replace([np.inf, -np.inf], np.nan).fillna(feature_medians_sens)
    processed_sens_data[name] = df_clean

# 提取公共特征
all_features_sens = [set(df.columns) for df in processed_sens_data.values()]
common_features_sens = sorted(list(set.intersection(*all_features_sens) - {'status'}))

X_train_raw_sens = processed_sens_data['Sens_Train'][common_features_sens]
y_train_sens = processed_sens_data['Sens_Train']['status'].astype(int)

X_val_raw_sens = processed_sens_data['Sens_Validation'][common_features_sens]
y_val_sens = processed_sens_data['Sens_Validation']['status'].astype(int)

# 3. 特征标准化 (Fit 于新训练集)
ss_sens = StandardScaler()
X_train_scaled_sens = ss_sens.fit_transform(X_train_raw_sens)
X_val_scaled_sens = ss_sens.transform(X_val_raw_sens)

# 4. SMOTE 平衡处理 (仅应用于新训练集)
print(f"原敏感性训练集分布 -> 0: {np.sum(y_train_sens == 0)}, 1: {np.sum(y_train_sens == 1)}")
smote_sens = SMOTE(random_state=42)
X_train_res_sens, y_train_res_sens = smote_sens.fit_resample(X_train_scaled_sens, y_train_sens)
print(f"SMOTE 后分布 -> 0: {np.sum(y_train_res_sens == 0)}, 1: {np.sum(y_train_res_sens == 1)}\n")

# 5. 定义并训练 XGBoost 模型 (使用我们之前搜索确定的参数)
xgb_params = {
    'subsample': 0.6, 'n_estimators': 600, 'max_depth': 6, 
    'learning_rate': 0.01, 'gamma': 0.5,
    'objective': 'binary:logistic', 'eval_metric': 'auc', 
    'random_state': 42, 'n_jobs': -1
}

print("🚀 正在敏感性数据集上训练 XGBoost 模型...")
xgb_model_sens = xgb.XGBClassifier(**xgb_params)
xgb_model_sens.fit(X_train_res_sens, y_train_res_sens)

# 6. 提取预测概率 (使用标准化后的真实分布数据)
y_train_probs_sens = xgb_model_sens.predict_proba(X_train_scaled_sens)[:, 1]
y_val_probs_sens = xgb_model_sens.predict_proba(X_val_scaled_sens)[:, 1]

# 7. 计算包含 Bootstrap 的 95% 置信区间
def calculate_auc_with_ci(y_true, y_probs, n_bootstraps=1000):
    y_true = np.array(y_true)
    point_auc = roc_auc_score(y_true, y_probs)
    
    bootstrapped_aucs = []
    n_samples = len(y_true)
    
    for _ in range(n_bootstraps):
        indices = resample(np.arange(n_samples), replace=True)
        y_true_b = y_true[indices]
        y_probs_b = y_probs[indices]
        
        if len(np.unique(y_true_b)) < 2:
            continue
            
        bootstrapped_aucs.append(roc_auc_score(y_true_b, y_probs_b))
        
    lower_bound = np.percentile(bootstrapped_aucs, 2.5)
    upper_bound = np.percentile(bootstrapped_aucs, 97.5)
    
    return point_auc, lower_bound, upper_bound

print("⏳ 正在进行 Bootstrap 重抽样计算 AUC 的 95% 置信区间 (约需几十秒)...")
train_auc, train_lower, train_upper = calculate_auc_with_ci(y_train_sens, y_train_probs_sens)
val_auc, val_lower, val_upper = calculate_auc_with_ci(y_val_sens, y_val_probs_sens)

# 8. 打印最终结果
print("\n" + "="*65)
print(f"📊 基于新数据集的 XGBoost 敏感性分析结果 (包含 95% CI)")
print("="*65)
print(f"【敏感性训练集】")
print(f"   -> AUC: {train_auc:.4f} (95% CI: {train_lower:.4f} - {train_upper:.4f})\n")

print(f"【敏感性验证集】")
print(f"   -> AUC: {val_auc:.4f} (95% CI: {val_lower:.4f} - {val_upper:.4f})")
print("="*65)